In [ ]:
# AI Taxonomy Inference & Validation Engine

**Objective:**
This notebook executes hierarchical, zero-shot classification utilizing the **Gemma 4 (26B)** model. It tests the AI's ability to navigate the 3-tier enterprise taxonomy tree (`master_taxonomy.json`) to classify unstructured product strings.

**Validation Protocol:**
To rigorously test accuracy, this script samples verified records from the `Categorized.xlsx` (Mercari) dataset. Because these records already contain human-verified classifications (`category_1`, `category_2`, `category_3`), we can dynamically prompt the model using only the `item_name`, and mathematically evaluate its predicted path against the known "ground truth" path.

In [ ]:
# Install required dependencies
!pip install -q transformers torch pandas openpyxl huggingface_hub

import os
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Configuration & File Paths ---
# INSTRUCTION: Update these placeholder paths.
INPUT_DIR = 'path/to/your/output_directory/Split_Files'
EXCEL_PATH = os.path.join(INPUT_DIR, 'Categorized.xlsx')
TAXONOMY_PATH = os.path.join(INPUT_DIR, 'master_taxonomy.json')
OUTPUT_REPORT_PATH = os.path.join(INPUT_DIR, 'Validation_Report.xlsx')

# Target Columns
TITLE_COLUMN = "item_name"
TRUE_C1_COL = "category_1"
TRUE_C2_COL = "category_2"
TRUE_C3_COL = "category_3"

# Hugging Face Authentication (Required for Gemma models)
# INSTRUCTION: Insert your Hugging Face access token here safely
HF_TOKEN = "YOUR_HUGGINGFACE_ACCESS_TOKEN_HERE"

# --- Load Enterprise Assets ---
print("Loading taxonomy and reference dataset...")
try:
    with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
        taxonomy = json.load(f)
    df = pd.read_excel(EXCEL_PATH)
    print(f"✅ Loaded taxonomy tree and {len(df)} reference products.")
except Exception as e:
    print(f"❌ Initialization Error: {e}")

# --- Initialize Gemma 4 26B ---
print("Initializing Gemma 4 (26B)... This requires significant VRAM.")
model_id = "google/gemma-4-26b"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
# Utilizing torch.bfloat16 and automatic device mapping for large model efficiency
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16, 
    device_map="auto", 
    token=HF_TOKEN
)
print("✅ Model loaded and ready for inference.")

In [ ]:
# --- The AI Inference Engine ---

def ask_gemma_category(product: str, options: list, level_name: str, parent_context: str = "") -> str:
    """Constructs a deterministic prompt and queries Gemma 4 for a single category string."""
    
    options_list_str = "\n".join([f"- {opt}" for opt in options])
    context_str = f"Parent Category Path: {parent_context}\n" if parent_context else ""
    
    system_instruction = (
        "You are a strict data classification pipeline.\n"
        "Output ONLY the exact category name from the provided 'Valid Categories' list.\n"
        "NO explanations. NO conversational text."
    )
    
    user_prompt = (
        f"{context_str}Valid {level_name} Categories:\n{options_list_str}\n\n"
        f"Task: Classify this product into exactly one of the categories above.\n"
        f"Product: \"{product}\"\nCategory:"
    )

    # Format using Gemma's native chat template
    messages = [
        {"role": "user", "content": f"{system_instruction}\n\n{user_prompt}"}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Highly deterministic generation parameters
    outputs = model.generate(
        **inputs, 
        max_new_tokens=20, 
        temperature=0.01, # Near-zero for strict determinism 
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    
    # Clean output string
    prediction = response.replace('"', '').replace("'", "").lstrip('- ').strip()
    return prediction

def classify_product_cascading(product: str, taxonomy_tree: dict) -> tuple:
    """Executes the 3-tier recursive classification logic with validation fail-safes."""
    cat1, cat2, cat3 = "Other", "Other", "Other"
    
    # LEVEL 1
    l1_options = list(taxonomy_tree.keys())
    l1_pred = ask_gemma_category(product, l1_options, "Level 1")
    
    if l1_pred in taxonomy_tree:
        cat1 = l1_pred
        
        # LEVEL 2
        l2_data = taxonomy_tree[cat1]
        l2_options = list(l2_data.keys()) if isinstance(l2_data, dict) else l2_data
        l2_pred = ask_gemma_category(product, l2_options, "Level 2", cat1)
        
        if (isinstance(l2_data, list) and l2_pred in l2_data) or (isinstance(l2_data, dict) and l2_pred in l2_data):
            cat2 = l2_pred
            
            # LEVEL 3
            if isinstance(l2_data, dict) and isinstance(l2_data[cat2], (list, dict)):
                l3_data = l2_data[cat2]
                l3_options = list(l3_data.keys()) if isinstance(l3_data, dict) else l3_data
                
                if len(l3_options) > 0:
                    parent_path = f"{cat1} > {cat2}"
                    l3_pred = ask_gemma_category(product, l3_options, "Level 3", parent_path)
                    
                    if (isinstance(l3_data, list) and l3_pred in l3_data) or (isinstance(l3_data, dict) and l3_pred in l3_data):
                        cat3 = l3_pred

    return cat1, cat2, cat3

In [ ]:
# --- Single Product Validation Test ---

print("=== Single Item Diagnostic Validation ===")
random_row = df.sample(1).iloc[0]

product_name = str(random_row[TITLE_COLUMN])
true_path = [str(random_row[TRUE_C1_COL]), str(random_row[TRUE_C2_COL]), str(random_row[TRUE_C3_COL])]

print(f"📦 Product: {product_name}")
print(f"✅ Ground Truth: {' > '.join(true_path)}")

print("\n🤖 Initiating Gemma 4 Inference...")
pred_c1, pred_c2, pred_c3 = classify_product_cascading(product_name, taxonomy)
pred_path = [pred_c1, pred_c2, pred_c3]

print(f"🎯 AI Prediction: {' > '.join(pred_path)}")

# Evaluation
if pred_path == true_path:
    print("\n🟢 VERDICT: Perfect Match. Inference successful.")
else:
    print("\n🔴 VERDICT: Mismatch detected. Model deviated from truth table.")

In [ ]:
# --- Batch Validation Testing (10 Items) ---

print("\n=== Initiating Batch Validation Engine ===")
sample_size = 10
sample_df = df.sample(sample_size)
validation_results = []

for index, row in sample_df.iterrows():
    product_name = str(row[TITLE_COLUMN])
    true_c1, true_c2, true_c3 = str(row[TRUE_C1_COL]), str(row[TRUE_C2_COL]), str(row[TRUE_C3_COL])
    
    print(f"Analyzing: {product_name[:40]}...")
    pred_c1, pred_c2, pred_c3 = classify_product_cascading(product_name, taxonomy)
    
    # Calculate binary accuracy flags
    match_l1 = (true_c1 == pred_c1)
    match_l2 = (true_c2 == pred_c2)
    match_l3 = (true_c3 == pred_c3)
    perfect_match = match_l1 and match_l2 and match_l3
    
    validation_results.append({
        "item_name": product_name,
        "true_L1": true_c1, "pred_L1": pred_c1, "match_L1": match_l1,
        "true_L2": true_c2, "pred_L2": pred_c2, "match_L2": match_l2,
        "true_L3": true_c3, "pred_L3": pred_c3, "match_L3": match_l3,
        "is_perfect_match": perfect_match
    })

# Export Validation Matrix
print("\n💾 Packaging telemetry and exporting to Excel...")
output_df = pd.DataFrame(validation_results)
output_df.to_excel(OUTPUT_REPORT_PATH, index=False)

# Print Summary Metrics
total_perfect = output_df['is_perfect_match'].sum()
print(f"\n📊 VALIDATION SUMMARY:")
print(f"Total Processed: {sample_size}")
print(f"Perfect Matches (All 3 Levels): {total_perfect} ({total_perfect/sample_size * 100}%)")
print(f"Report exported to: {OUTPUT_REPORT_PATH}")